# KYC Thesis — Model 2: Face Embedding (Inference-First, Project Scope)

Core workflow (default):
- FP32 vs INT8 vs optional Distilled comparison
- Single-seed evaluation for fast progress runs
- Drive progress + plots + ONNX export + matrix logging

Optional final-thesis extension (disabled by default):
- Multi-seed evaluation (mean ± std)
- Bootstrap 95% CI
- Paired-bootstrap significance test


In [ ]:
%%capture
!pip install -q facenet-pytorch==2.6.0 timm==1.0.8 scikit-learn==1.5.1 onnx==1.16.2 onnxruntime==1.18.1


In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image

from facenet_pytorch import InceptionResnetV1
from sklearn.metrics import roc_curve, auc

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/kyc_thesis')
RUN_ID = f'face_rigorous_{int(time.time())}'
RUN_DIR = BASE_DIR / 'experiments' / 'face' / RUN_ID
REPORT_DIR = RUN_DIR / 'reports'
EXPORT_DIR = RUN_DIR / 'exports'
CKPT_DIR = RUN_DIR / 'checkpoints'
MATRIX_CSV = BASE_DIR / 'reports' / 'comparison_matrix.csv'

for d in [RUN_DIR, REPORT_DIR, EXPORT_DIR, CKPT_DIR, BASE_DIR / 'reports']:
    d.mkdir(parents=True, exist_ok=True)

PROGRESS_JSON = RUN_DIR / 'progress.json'

def save_progress(stage, status='done', **extra):
    payload = {'run_id': RUN_ID, 'stage': stage, 'status': status, 'time': int(time.time())}
    payload.update(extra)
    PROGRESS_JSON.write_text(json.dumps(payload, indent=2))

save_progress('init')
print('RUN_DIR:', RUN_DIR)


In [ ]:
RUN_TRAIN_DISTILL = False
RUN_MULTI_SEED = False
RUN_ADVANCED_STATS = False

PRIMARY_SEED = 42
SEEDS = [42, 43, 44] if RUN_MULTI_SEED else [PRIMARY_SEED]

BATCH_SIZE = 64
EPOCHS_DISTILL = 5
LR = 1e-4

PAIRS_CSV = BASE_DIR / 'data' / 'face' / 'pairs.csv'
assert PAIRS_CSV.exists(), f'Missing: {PAIRS_CSV}'
pairs_df = pd.read_csv(PAIRS_CSV)
assert {'img_a','img_b','label'}.issubset(pairs_df.columns)

print('Pairs:', len(pairs_df))
print(pairs_df['label'].value_counts())
print('Seeds:', SEEDS)
save_progress('pairs_loaded', n_pairs=len(pairs_df), seeds=SEEDS, run_multi_seed=RUN_MULTI_SEED)


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def load_img(path, size=160):
    p = Path(path)
    if not p.is_absolute():
        p = BASE_DIR / path
    img = Image.open(p).convert('RGB').resize((size,size))
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - 0.5) / 0.5
    x = torch.tensor(arr).permute(2,0,1).unsqueeze(0)
    return x

def cosine(a,b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a,b))

teacher = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)

class StudentEmbedder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_small_050', pretrained=True, num_classes=0, global_pool='avg')
        in_dim = self.backbone.num_features
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        f = self.backbone(x)
        z = self.proj(f)
        return F.normalize(z, dim=1)

def eval_embeddings(embed_fn, seed):
    set_seed(seed)
    scores, labels = [], []
    with torch.no_grad():
        for _, r in pairs_df.iterrows():
            xa = load_img(r['img_a']).to(DEVICE)
            xb = load_img(r['img_b']).to(DEVICE)
            ea = embed_fn(xa)
            eb = embed_fn(xb)
            if torch.is_tensor(ea):
                ea = ea.detach().cpu().numpy().squeeze()
            if torch.is_tensor(eb):
                eb = eb.detach().cpu().numpy().squeeze()
            scores.append(cosine(ea, eb))
            labels.append(int(r['label']))

    y = np.array(labels)
    s = np.array(scores)
    fpr, tpr, thr = roc_curve(y, s)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    eer = float((fpr[idx] + fnr[idx]) / 2.0)
    roc_auc = float(auc(fpr, tpr))
    pred = (s >= thr[idx]).astype(int)
    acc = float((pred == y).mean())
    return {'eer': eer, 'auc': roc_auc, 'acc': acc, 'scores': s, 'labels': y, 'thr': float(thr[idx]), 'pred': pred, 'fpr': fpr, 'tpr': tpr}

def teacher_embed(x):
    return F.normalize(teacher(x), dim=1)

def latency_ms(embed_fn, runs=80):
    x = torch.randn(1,3,160,160).to(DEVICE)
    with torch.no_grad():
        for _ in range(10):
            _ = embed_fn(x)
    ts=[]
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = embed_fn(x)
            ts.append((time.perf_counter()-t0)*1000)
    return float(np.mean(ts))

def state_size_mb(model_obj):
    p = CKPT_DIR / 'tmp_size_face.pth'
    torch.save(model_obj.state_dict(), p)
    s = p.stat().st_size / 1e6
    p.unlink(missing_ok=True)
    return float(s)


In [ ]:
# Optional distillation dataset from unique image paths in pairs.csv
class DistillImageDataset(Dataset):
    def __init__(self, img_paths):
        self.paths = list(img_paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = Path(self.paths[idx])
        if not p.is_absolute():
            p = BASE_DIR / p
        img = Image.open(p).convert('RGB').resize((160,160))
        arr = np.asarray(img).astype(np.float32)/255.0
        arr = (arr - 0.5)/0.5
        x = torch.tensor(arr).permute(2,0,1)
        return x

def train_student(seed):
    set_seed(seed)
    student = StudentEmbedder().to(DEVICE)
    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=1e-4)

    uniq = pd.unique(pd.concat([pairs_df['img_a'], pairs_df['img_b']], ignore_index=True))
    ds = DistillImageDataset(uniq)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    best = 1e9
    path = CKPT_DIR / f'face_distilled_seed{seed}.pt'

    for epoch in range(1, EPOCHS_DISTILL + 1):
        student.train()
        total = 0.0
        n = 0
        for x in dl:
            x = x.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.no_grad():
                t = F.normalize(teacher(x), dim=1)
            s = student(x)
            loss = F.mse_loss(s, t)
            loss.backward()
            opt.step()
            total += loss.item() * x.size(0)
            n += x.size(0)

        avg = total / max(n, 1)
        if avg < best:
            best = avg
            torch.save(student.state_dict(), path)

    student.load_state_dict(torch.load(path, map_location=DEVICE))
    return student, path


In [ ]:
rows = []
paired_preds = {'fp32': [], 'int8': [], 'distilled': []}
paired_labels = []

for seed in SEEDS:
    set_seed(seed)

# FP32 teacher
    fp = eval_embeddings(teacher_embed, seed)
    fp_lat = latency_ms(teacher_embed)
    fp_sz = state_size_mb(teacher)
    rows.append({'seed': seed, 'variant': 'fp32', 'acc': fp['acc'], 'auc': fp['auc'], 'eer': fp['eer'], 'latency_ms': fp_lat, 'size_mb': fp_sz})

# INT8 dynamic quantized teacher (Linear layers only)
    t_q = torch.quantization.quantize_dynamic(teacher.cpu(), {nn.Linear}, dtype=torch.qint8).to(DEVICE).eval()
    def t_q_embed(x):
        return F.normalize(t_q(x), dim=1)
    i8 = eval_embeddings(t_q_embed, seed)
    i8_lat = latency_ms(t_q_embed)
    i8_sz = state_size_mb(t_q)
    rows.append({'seed': seed, 'variant': 'int8', 'acc': i8['acc'], 'auc': i8['auc'], 'eer': i8['eer'], 'latency_ms': i8_lat, 'size_mb': i8_sz})

# Distilled student
    dist_path = CKPT_DIR / f'face_distilled_seed{seed}.pt'
    if RUN_TRAIN_DISTILL or dist_path.exists():
        if RUN_TRAIN_DISTILL and not dist_path.exists():
            student, dist_path = train_student(seed)
        else:
            student = StudentEmbedder().to(DEVICE)
            student.load_state_dict(torch.load(dist_path, map_location=DEVICE))

        def s_embed(x):
            return student(x)
        ds = eval_embeddings(s_embed, seed)
        ds_lat = latency_ms(s_embed)
        ds_sz = state_size_mb(student)
        rows.append({'seed': seed, 'variant': 'distilled', 'acc': ds['acc'], 'auc': ds['auc'], 'eer': ds['eer'], 'latency_ms': ds_lat, 'size_mb': ds_sz})

    paired_labels.append(fp['labels'])
    paired_preds['fp32'].append(fp['pred'])
    paired_preds['int8'].append(i8['pred'])
    if 'ds' in locals():
        paired_preds['distilled'].append(ds['pred'])

    save_progress('seed_done', seed=seed)

rdf = pd.DataFrame(rows)
rdf.to_csv(REPORT_DIR / 'per_seed_metrics.csv', index=False)
display(rdf)


In [ ]:
summary = rdf.groupby('variant').agg(['mean','std'])
summary.columns = ['_'.join(c) for c in summary.columns]
summary = summary.reset_index()
summary.to_csv(REPORT_DIR / 'summary_mean_std.csv', index=False)

display(summary)
save_progress('summary_saved')

if RUN_ADVANCED_STATS:
    def bootstrap_ci(values, n=2000, alpha=0.05):
        vals = np.array(values)
        boots = []
        for _ in range(n):
            sample = np.random.choice(vals, size=len(vals), replace=True)
            boots.append(sample.mean())
        lo = float(np.percentile(boots, 100 * alpha / 2))
        hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
        return lo, hi

    ci_rows = []
    for v in rdf['variant'].unique():
        vv = rdf[rdf['variant']==v]
        ci_rows.append({
          'variant': v,
          'acc_ci95': bootstrap_ci(vv['acc']),
          'auc_ci95': bootstrap_ci(vv['auc']),
          'eer_ci95': bootstrap_ci(vv['eer'])
        })
    ci_df = pd.DataFrame(ci_rows)
    ci_df.to_csv(REPORT_DIR / 'summary_ci95.csv', index=False)
    display(ci_df)
    save_progress('advanced_ci_saved')
else:
    print('Advanced CI skipped (RUN_ADVANCED_STATS=False).')
    save_progress('advanced_ci_skipped')


In [ ]:
if RUN_ADVANCED_STATS:
    # Paired bootstrap significance on EER difference using first seed predictions
    y = paired_labels[0]
    p_fp = paired_preds['fp32'][0]
    p_i8 = paired_preds['int8'][0]

    def err_rate(y_true, pred):
        return float(np.mean(y_true != pred))

    def paired_bootstrap_p(y_true, pred_a, pred_b, n=2000):
        idx = np.arange(len(y_true))
        diffs = []
        for _ in range(n):
            s = np.random.choice(idx, size=len(idx), replace=True)
            da = err_rate(y_true[s], pred_a[s])
            db = err_rate(y_true[s], pred_b[s])
            diffs.append(db - da)
        diffs = np.array(diffs)
        p = 2 * min(np.mean(diffs >= 0), np.mean(diffs <= 0))
        return float(p), float(np.mean(diffs))

    p_fp_i8, d_fp_i8 = paired_bootstrap_p(y, p_fp, p_i8)
    sig = [{'comparison': 'fp32_vs_int8', 'paired_bootstrap_p': p_fp_i8, 'mean_err_diff': d_fp_i8}]

    if len(paired_preds['distilled']) > 0:
        p_ds = paired_preds['distilled'][0]
        p_fp_ds, d_fp_ds = paired_bootstrap_p(y, p_fp, p_ds)
        sig.append({'comparison': 'fp32_vs_distilled', 'paired_bootstrap_p': p_fp_ds, 'mean_err_diff': d_fp_ds})

    sig_df = pd.DataFrame(sig)
    sig_df.to_csv(REPORT_DIR / 'significance_paired_bootstrap.csv', index=False)
    display(sig_df)
    save_progress('significance_saved')
else:
    print('Significance test skipped (RUN_ADVANCED_STATS=False).')
    save_progress('significance_skipped')


In [ ]:
# Save plots + ONNX + matrix rows
best_auc = rdf.sort_values('auc', ascending=False).iloc[0]

# Plot aggregate EER by variant
plt.figure(figsize=(7,4))
import seaborn as sns
sns.barplot(data=rdf, x='variant', y='eer', estimator=np.mean, errorbar='sd')
plt.title('Face EER by Variant (mean ± sd over seeds)')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'eer_variant_bar.png', dpi=140)

onnx_path = EXPORT_DIR / 'face_embedder_fp32.onnx'
dummy = torch.randn(1,3,160,160)
teacher_cpu = teacher.cpu().eval()
torch.onnx.export(teacher_cpu, dummy, str(onnx_path), input_names=['input'], output_names=['embedding'], opset_version=13)

matrix_cols = ['timestamp_utc','task','run_id','variant','dataset_train','dataset_eval','model_name','epochs','batch_size','img_size','seed','accuracy','f1','auc','eer','apcer','bpcer','acer','latency_ms','model_size_mb','artifact_path','notes']
if MATRIX_CSV.exists():
    mdf = pd.read_csv(MATRIX_CSV)
else:
    mdf = pd.DataFrame(columns=matrix_cols)

for v in rdf['variant'].unique():
    vv = rdf[rdf['variant']==v]
    mdf = pd.concat([mdf, pd.DataFrame([{
      'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
      'task':'face', 'run_id':RUN_ID, 'variant':v,
      'dataset_train':'vggface2_pretrained_plus_optional_distill', 'dataset_eval':'pairs_csv',
      'model_name':'inceptionresnet_or_student', 'epochs':EPOCHS_DISTILL if v=='distilled' else 0,
      'batch_size':BATCH_SIZE, 'img_size':160, 'seed':'multi',
      'accuracy':vv['acc'].mean(), 'auc':vv['auc'].mean(), 'eer':vv['eer'].mean(),
      'latency_ms':vv['latency_ms'].mean(), 'model_size_mb':vv['size_mb'].mean(),
      'artifact_path':str(RUN_DIR), 'notes':'mean over seeds'
    }])], ignore_index=True)

mdf.to_csv(MATRIX_CSV, index=False)
save_progress('complete', onnx=str(onnx_path), matrix=str(MATRIX_CSV), best_variant=str(best_auc['variant']))
print('Complete. ONNX:', onnx_path)
